# 資料處理

## 檢查版本

In [2]:
import sys, xarray as xr
print("Python exe:", sys.executable)
print("Engines:", xr.backends.list_engines())

Python exe: /home/jundian/miniconda3/envs/geospatial-neural-adapter/bin/python
Engines: {'netcdf4': <NetCDF4BackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using netCDF4 in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.NetCDF4BackendEntrypoint.html, 'h5netcdf': <H5netcdfBackendEntrypoint>
  Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using h5netcdf in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.H5netcdfBackendEntrypoint.html, 'scipy': <ScipyBackendEntrypoint>
  Open netCDF files (.nc, .nc4, .cdf and .gz) using scipy in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html, 'store': <StoreBackendEntrypoint>
  Open AbstractDataStore instances in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html}


## 讀nc4

In [3]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm


def check_data_folder(folder: str) -> bool:
    return os.path.exists(folder) and os.path.isdir(folder)


def load_data(file_path: str) -> xr.Dataset:
    """
    Load data from a NetCDF file, trying netcdf4 then h5netcdf.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    # 優先 netcdf4
    try:
        return xr.open_dataset(file_path, engine="netcdf4")
    except Exception as e1:
        # 改試 h5netcdf
        try:
            return xr.open_dataset(file_path, engine="h5netcdf")
        except Exception as e2:
            raise RuntimeError(
                f"Failed to open {file_path} with netcdf4 and h5netcdf.\n"
                f"e1: {e1}\n"
                f"e2: {e2}\n"
                "請確認這個環境有安裝 netCDF4 或 h5netcdf。"
            )


# ================= main program =================

data_folder = "nc4"
var_name = "T2M"  # 目標變數名稱（請確認檔內真的叫這個）

# 1. 檢查資料夾
if not check_data_folder(data_folder):
    raise FileNotFoundError(f"Data folder not found: {data_folder}")
print(f"Data folder found: {data_folder}")

# 2. 找出所有像 1980-01.nc4 的檔案
pattern = os.path.join(data_folder, "*.nc4")
file_list = sorted(glob.glob(pattern))

if len(file_list) == 0:
    raise FileNotFoundError(f"No nc4 files found with pattern: {pattern}")

print(f"Found {len(file_list)} files.")
print("First 5 files:", file_list[:5])

# 3. 用第一個檔案確認經緯度與變數存在，必要時自動偵測 var_name
with load_data(file_list[0]) as sample_data:
    print("Data variables in first file:")
    print(list(sample_data.data_vars))
    print("Coordinates:")
    print({k: sample_data[k].shape for k in sample_data.coords})

    # 找出所有長得像 (time, lat, lon) 的候選變數
    candidates = []
    for v in sample_data.data_vars:
        dims = set(sample_data[v].dims)
        if {"time", "lat", "lon"}.issubset(dims):
            candidates.append(v)

    # 如果原本設定的 var_name 不在，就試著自動改
    if var_name not in sample_data.data_vars:
        if len(candidates) == 1:
            auto_var = candidates[0]
            print(f"[Info] Variable '{var_name}' not found, auto-select '{auto_var}' as target.")
            var_name = auto_var
        else:
            raise KeyError(
                f"Variable '{var_name}' not found in file: {file_list[0]}\n"
                f"Available data_vars: {list(sample_data.data_vars)}\n"
                f"3D (time,lat,lon) candidates: {candidates}\n"
                "請將上方其中一個正確變數名稱填入 var_name。"
            )

    # 取經緯度
    if "lat" not in sample_data.coords or "lon" not in sample_data.coords:
        raise KeyError("lat/lon coordinates not found in sample file.")
    lat = sample_data["lat"].values
    lon = sample_data["lon"].values

nlat = lat.shape[0]
nlon = lon.shape[0]
print(f"Confirmed var_name = {var_name}")
print(f"Lat: {nlat}, Lon: {nlon}")


# 4. 逐檔讀入，累積到 list
data_list = []
time_list = []

for f in tqdm(file_list, desc="Combining"):
    with load_data(f) as ds:
        da = ds[var_name]  # (time, lat, lon)

        # 確保 lat/lon 一致（保險，可視情況註解）
        if da.sizes["lat"] != nlat or da.sizes["lon"] != nlon:
            raise ValueError(f"Lat/Lon size mismatch in file: {f}")

        # 資料轉 float32，省記憶體
        data_list.append(da.values.astype(np.float32))

        if "time" not in ds:
            raise KeyError(f"'time' coordinate not found in file: {f}")

        # decode_cf 確保時間是真正 datetime
        t = xr.decode_cf(ds)["time"].values
        time_list.append(t.astype("datetime64[ns]"))

# 5. 串起來 → (ntot, lat, lon) & DatetimeIndex
combined = np.concatenate(data_list, axis=0)   # (ntot, nlat, nlon)
time_array = np.concatenate(time_list, axis=0) # (ntot,)

if combined.shape[0] != time_array.shape[0]:
    raise ValueError(
        f"time length ({time_array.shape[0]}) "
        f"!= data length ({combined.shape[0]})"
    )

# 依時間排序（通常已排序，這裡是保險）
sort_idx = np.argsort(time_array)
combined = combined[sort_idx]
time_array = time_array[sort_idx]

time_index = pd.to_datetime(time_array)

print(f"Combined data shape: {combined.shape}")
print(f"Time index: {time_index[0]} -> {time_index[-1]} (len={len(time_index)})")

# 6. 攤平成 cell × time
ntot, nlat, nlon = combined.shape
ncell = nlat * nlon

# (cell, time)
y_all = combined.reshape(ntot, ncell).T

# 建立每個 cell 的 (lon, lat)
lon_grid, lat_grid = np.meshgrid(lon, lat)
gg = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])  # (cell, 2)

print(f"y_all shape: {y_all.shape}  (cells x time)")
print(f"gg shape: {gg.shape}        (cells x [lon, lat])")


Data folder found: nc4
Found 548 files.
First 5 files: ['nc4/1980-01.nc4', 'nc4/1980-02.nc4', 'nc4/1980-03.nc4', 'nc4/1980-04.nc4', 'nc4/1980-05.nc4']
Data variables in first file:
['T2M', 'T2MDEW', 'Var_T2M', 'T2MWET']
Coordinates:
{'lon': (576,), 'time': (1,), 'lat': (361,)}
Confirmed var_name = T2M
Lat: 361, Lon: 576


Combining: 100%|██████████| 548/548 [00:15<00:00, 36.18it/s]


Combined data shape: (548, 361, 576)
Time index: 1980-01-01 00:30:00 -> 2025-08-01 00:30:00 (len=548)
y_all shape: (207936, 548)  (cells x time)
gg shape: (207936, 2)        (cells x [lon, lat])


# 限制範圍抽樣 in USA

In [4]:
# ===== 只看美國本土範圍，並從中抽樣 25 個格點 =====
import numpy as np

# gg: (ncell, 2)；第 0 欄是 lon，第 1 欄是 lat
lon_all = gg[:, 0].astype(float)
lat_all = gg[:, 1].astype(float)

# 如果經度是 0~360，轉成 -180~180（如果本來就是 -180~180，這個轉換也不會壞掉）
lon_all_180 = ((lon_all + 180) % 360) - 180

# 美國本土 48 州的大致範圍（可以再微調）
lon_min, lon_max = -125, -66
lat_min, lat_max = 24, 50

# 建立遮罩：只保留在美國本土範圍內的格點
mask_us_mainland = (
    (lon_all_180 >= lon_min) & (lon_all_180 <= lon_max) &
    (lat_all      >= lat_min) & (lat_all      <= lat_max)
)

idx_us_mainland = np.where(mask_us_mainland)[0]
print(f"美國本土範圍內的格點數量: {len(idx_us_mainland)}")

if len(idx_us_mainland) == 0:
    raise ValueError("在設定的美國本土範圍內沒有格點，請調整 lon_min/max 或 lat_min/max。")

# 只保留美國本土子集合
y_us = y_all[idx_us_mainland, :]  # (n_us, ntot)
coords_us = np.column_stack([lon_all_180[idx_us_mainland], lat_all[idx_us_mainland]])  # (n_us, 2)

print(f"美國本土子集合 y_us shape: {y_us.shape}")
print(f"美國本土子集合 coords_us shape: {coords_us.shape}")

# 從美國本土格點中隨機抽樣 25 個（如果不足 25，就全部使用）
n_sample = min(25, len(idx_us_mainland))
np.random.seed(42)  # 為了可重現
sample_idx_in_us = np.random.choice(len(idx_us_mainland), size=n_sample, replace=False)

# 這是「在原始格點編號中的 index」
sample_idx_global = idx_us_mainland[sample_idx_in_us]

# 抽樣後的 time series 與座標
y_sample = y_all[sample_idx_global, :]  # (n_sample, ntot)
coords_sample = np.column_stack([lon_all_180[sample_idx_global], lat_all[sample_idx_global]])  # (n_sample, 2)

# 如果之後畫圖想分開拿 lon / lat：
lon_us_sample = coords_sample[:, 0]
lat_us_sample = coords_sample[:, 1]

print(f"抽樣 {n_sample} 個美國本土格點")
print("前 5 個 sample index (global):", sample_idx_global[:5])
print("前 5 個 sample 座標 (lon, lat):")
for i in range(min(5, n_sample)):
    print(f"  #{i}: lon={lon_us_sample[i]:.2f}, lat={lat_us_sample[i]:.2f}")


美國本土範圍內的格點數量: 5035
美國本土子集合 y_us shape: (5035, 548)
美國本土子集合 coords_us shape: (5035, 2)
抽樣 25 個美國本土格點
前 5 個 sample index (global): [153335 159677 145263 156825 138939]
前 5 個 sample 座標 (lon, lat):
  #0: lon=-105.62, lat=43.00
  #1: lon=-101.88, lat=48.50
  #2: lon=-110.62, lat=36.00
  #3: lon=-84.38, lat=46.00
  #4: lon=-103.12, lat=30.50


## VARIMA

In [ ]:
# ===== VARIMA on the 25 sampled US grid cells =====
import numpy as np
import pandas as pd
from typing import Tuple
from darts import TimeSeries
from darts.models import VARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.api import VAR   # 用來做 VAR(p) 拿 AIC/BIC/HQIC/FPE

print("\n===== VARIMA baseline on 25 sampled US grid cells =====")

# 1. 建 target DataFrame：25 個格點 × T 時間點（原始單位）
SAMPLE_SIZE = y_sample.shape[0]   # 預期是 25
T_total = y_sample.shape[1]

ts_df = pd.DataFrame(
    y_sample.T,  # (T, SAMPLE_SIZE)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps:", T_total)
print("ts_df shape:", ts_df.shape)  # (T, SAMPLE_SIZE)

# 2. 切成 70 / 15 / 15（train / val / test）
train_frac = 0.7
val_frac   = 0.15   # test 也是 0.15

cut_train = int(T_total * train_frac)
cut_val   = int(T_total * (train_frac + val_frac))

idx_time = ts_df.index
split_1_time = idx_time[cut_train]
split_2_time = idx_time[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)

train_raw, tmp_raw = ts_all_raw.split_before(split_1_time)
val_raw,   test_raw = tmp_raw.split_before(split_2_time)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

# 3. 用 train 統計量標準化
train_df = ts_df.iloc[:cut_train]

mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)  # 避免除以 0

ts_df_scaled = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

train_ts, tmp_ts = ts_all_scaled.split_before(split_1_time)
val_ts,   test_ts = tmp_ts.split_before(split_2_time)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)
print("\n[Scaled TimeSeries lengths]")
print("T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 真實值（原單位，用來算 MSE/MAE/R²）
vals_all      = ts_df.to_numpy(dtype=np.float32)  # (T_total, SAMPLE_SIZE)
val_true_raw  = vals_all[cut_train:cut_val, :]    # (T_val,   SAMPLE_SIZE)
test_true_raw = vals_all[cut_val:, :]             # (T_test,  SAMPLE_SIZE)

def inverse_scale(x: np.ndarray) -> np.ndarray:
    """
    x: shape (T_horizon, C)
    回傳反標準化後的值，shape 相同
    """
    return x * std_vec.values + mean_vec.values

# ======================================================
# 3.5 自己 loop VAR(p)，建立 AIC / BIC / HQIC / FPE table，並選階
# ======================================================
print("\n[Order selection by manual VAR(p) grid, with AIC/BIC/HQIC/FPE]")

# 月均資料：明確指定時間頻率為每月月初 MS（如果你是月底就改成 freq='M'）
train_scaled_df = ts_df_scaled.iloc[:cut_train].astype("float64").copy()
train_scaled_df.index = pd.date_range(
    start=train_scaled_df.index[0],
    periods=len(train_scaled_df),
    freq="MS",   # 月初頻率；若你的 index 是月底，改成 "M"
)

var_model = VAR(train_scaled_df)

desired_maxlags = 12

# 先找「最大可估的 lag」
max_feasible = None
for p in range(desired_maxlags, 0, -1):
    try:
        _ = var_model.fit(p)
        max_feasible = p
        print(f"最大可估的 lag = {p}")
        break
    except ValueError as e:
        print(f"lag = {p} 估計失敗（{e}），改試小一點的 lag...")

if max_feasible is None:
    raise RuntimeError("在 p=1..12 都無法估計 VAR(p)，可能樣本數太小或資料有問題。")

# 針對 p = 1..max_feasible，一個一個 fit，收集 IC
rows = []
for p in range(1, max_feasible + 1):
    try:
        res_p = var_model.fit(p)
        rows.append({
            "lag":  p,
            "AIC":  float(res_p.aic),
            "BIC":  float(res_p.bic),
            "HQIC": float(res_p.hqic),
            "FPE":  float(res_p.fpe),
        })
        print(f"  p={p} OK: AIC={res_p.aic:.4f}, BIC={res_p.bic:.4f}, HQIC={res_p.hqic:.4f}, FPE={res_p.fpe:.4e}")
    except ValueError as e:
        print(f"  p={p} 估計失敗（{e}），停止往更大 p 試。")
        break

ic_table = pd.DataFrame(rows)

# ---- 在 table 中用粗體標出每個指標的最佳（最小）值 ----
# 找出各指標的最佳 index（用數值的 ic_table）
best_idx_aic  = ic_table["AIC"].idxmin()
best_idx_bic  = ic_table["BIC"].idxmin()
best_idx_hqic = ic_table["HQIC"].idxmin()
best_idx_fpe  = ic_table["FPE"].idxmin()

# 建一個純字串版的 printed_table，避免 dtype 警告
printed_table = pd.DataFrame({
    "lag":  ic_table["lag"].astype(int).astype(str),
    "AIC":  [f"{v:.4f}" for v in ic_table["AIC"]],
    "BIC":  [f"{v:.4f}" for v in ic_table["BIC"]],
    "HQIC": [f"{v:.4f}" for v in ic_table["HQIC"]],
    "FPE":  [f"{v:.4e}" for v in ic_table["FPE"]],
})

# 在最佳位置加粗體標記（markdown 風格）
printed_table.loc[best_idx_aic,  "AIC"]  = printed_table.loc[best_idx_aic,  "AIC"]  + "*"
printed_table.loc[best_idx_bic,  "BIC"]  = printed_table.loc[best_idx_bic,  "BIC"]  + "*"
printed_table.loc[best_idx_hqic, "HQIC"] = printed_table.loc[best_idx_hqic, "HQIC"] + "*"
printed_table.loc[best_idx_fpe,  "FPE"]  = printed_table.loc[best_idx_fpe,  "FPE"]  + "*"

print("\nInformation Criteria table (p = 1..{}):".format(int(ic_table['lag'].max())))
print(printed_table.to_string(index=False))

# 用 BIC 選 lag（這裡還是用數值 ic_table）
p_selected = int(ic_table.loc[best_idx_bic, "lag"])
print(f"\nSelected lag by BIC = {p_selected}")

ORDER = (p_selected, 0, 0)
print(f"Final VARIMA ORDER = {ORDER}")

# ======================================================
# 4. VARIMA：train → predict val（用選出的 ORDER）
# ======================================================
print(f"\n[Step] VARIMA: train → predict VAL with ORDER={ORDER}")

model_varima_val = VARIMA(*ORDER)
model_varima_val.fit(train_ts)

pred_var_val = model_varima_val.predict(n=T_val)   # 預測 val 長度
pred_var_val_vals = np.asarray(pred_var_val.all_values(copy=False))
# shape 可能是 (T_val, C, 1)，把最後一維壓掉
if pred_var_val_vals.ndim == 3:
    pred_var_val_vals = pred_var_val_vals[..., 0]

pred_var_val_raw = inverse_scale(pred_var_val_vals)   # (T_val, SAMPLE_SIZE)

# ======================================================
# 5. VARIMA：train+val → predict test（同一個 ORDER）
# ======================================================
print(f"[Step] VARIMA: train+val → predict TEST with ORDER={ORDER}")

train_val_ts = train_ts.concatenate(val_ts)

model_varima_test = VARIMA(*ORDER)
model_varima_test.fit(train_val_ts)

pred_var_test = model_varima_test.predict(n=T_test)
pred_var_test_vals = np.asarray(pred_var_test.all_values(copy=False))
if pred_var_test_vals.ndim == 3:
    pred_var_test_vals = pred_var_test_vals[..., 0]

pred_var_test_raw = inverse_scale(pred_var_test_vals)  # (T_test, SAMPLE_SIZE)

# ======================================================
# 6. 計算 VAL / TEST 的 MSE / MAE / RMSE / R²
# ======================================================
y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
y_val_pred  = pred_var_val_raw.reshape(-1, SAMPLE_SIZE)
y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
y_test_pred = pred_var_test_raw.reshape(-1, SAMPLE_SIZE)

mse_var_val  = mean_squared_error(y_val_true,  y_val_pred)
mae_var_val  = mean_absolute_error(y_val_true, y_val_pred)
mse_var_test = mean_squared_error(y_test_true, y_test_pred)
mae_var_test = mean_absolute_error(y_test_true, y_test_pred)

rmse_var_val  = np.sqrt(mse_var_val)
rmse_var_test = np.sqrt(mse_var_test)

r2_var_val  = r2_score(y_val_true,  y_val_pred)
r2_var_test = r2_score(y_test_true, y_test_pred)

print("\n===== VARIMA RESULTS (25 sampled US cells) =====")
print(f"VAL  RMSE: {rmse_var_val:.4f}, MSE: {mse_var_val:.4f}, MAE: {mae_var_val:.4f}, R²: {r2_var_val:.4f}")
print(f"TEST RMSE: {rmse_var_test:.4f}, MSE: {mse_var_test:.4f}, MAE: {mae_var_test:.4f}, R²: {r2_var_test:.4f}")

metrics_table_varima = pd.DataFrame({
    "Model": ["VARIMA", "VARIMA"],
    "Split": ["VAL",     "TEST"],
    "RMSE":  [rmse_var_val,  rmse_var_test],
    "MSE":   [mse_var_val,   mse_var_test],
    "MAE":   [mae_var_val,   mae_var_test],
    "R2":    [r2_var_val,    r2_var_test],
})

print("\nMSE / MAE / RMSE / R² table (VARIMA, 25 cells):")
print(metrics_table_varima.to_string(index=False))


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



===== VARIMA baseline on 25 sampled US grid cells =====
Total time steps: 548
ts_df shape: (548, 25)
Train len (raw): 383, Val len: 82, Test len: 83

[Scaled TimeSeries lengths]
T_train = 383 T_val = 82 T_test = 83

[Order selection by manual VAR(p) grid, with AIC/BIC/HQIC/FPE]
最大可估的 lag = 12
  p=1 OK: AIC=-110.8640, BIC=-104.1506, HQIC=-108.2006, FPE=7.1560e-49
  p=2 OK: AIC=-110.6464, BIC=-97.4520, HQIC=-105.4114, FPE=9.2133e-49
  p=3 OK: AIC=-110.0193, BIC=-90.3185, HQIC=-102.2020, FPE=1.8991e-48
  p=4 OK: AIC=-109.3273, BIC=-83.0943, HQIC=-98.9169, FPE=4.6016e-48
  p=5 OK: AIC=-108.4244, BIC=-75.6336, HQIC=-95.4102, FPE=1.5827e-47
  p=6 OK: AIC=-108.3436, BIC=-68.9690, HQIC=-92.7149, FPE=2.9019e-47
  p=7 OK: AIC=-108.0995, BIC=-62.1149, HQIC=-89.8452, FPE=8.1543e-47
  p=8 OK: AIC=-108.3431, BIC=-55.7223, HQIC=-87.4524, FPE=2.0180e-46
  p=9 OK: AIC=-109.7795, BIC=-50.4960, HQIC=-86.2412, FPE=2.5054e-46
  p=10 OK: AIC=-111.6824, BIC=-45.7095, HQIC=-85.4855, FPE=4.0293e-46
  p=11 OK:

/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[Step] VARIMA: train+val → predict TEST with ORDER=(1, 0, 0)


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



===== VARIMA RESULTS (25 sampled US cells) =====
VAL  RMSE: 4.7002, MSE: 22.0919, MAE: 3.5191, R²: 0.5981
TEST RMSE: 3.9482, MSE: 15.5882, MAE: 2.8952, R²: 0.6850

MSE / MAE / RMSE / R² table (VARIMA, 25 cells):
 Model Split     RMSE       MSE      MAE       R2
VARIMA   VAL 4.700206 22.091938 3.519129 0.598139
VARIMA  TEST 3.948188 15.588186 2.895220 0.684964


### VARIMA Results — 25 Sampled US Grid Cells

| Model  | Split |   RMSE   |    MSE    |   MAE   |    R²    |
|:------:|:-----:|:--------:|:---------:|:-------:|:--------:|
| VARIMA |  VAL  | 4.700206 | 22.091938 | 3.519129 | 0.598139 |
| VARIMA | TEST  | 3.948188 | 15.588186 | 2.895220 | 0.684964 |


### VARIMA (+AIC)

In [ ]:
# ===== VARIMA on the 25 sampled US grid cells =====
import numpy as np
import pandas as pd
from typing import Tuple
from darts import TimeSeries
from darts.models import VARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.api import VAR   # 用來做 VAR(p) 拿 AIC/BIC/HQIC/FPE

print("\n===== VARIMA baseline on 25 sampled US grid cells =====")

# 1. 建 target DataFrame：25 個格點 × T 時間點（原始單位）
SAMPLE_SIZE = y_sample.shape[0]   # 預期是 25
T_total = y_sample.shape[1]

ts_df = pd.DataFrame(
    y_sample.T,  # (T, SAMPLE_SIZE)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps:", T_total)
print("ts_df shape:", ts_df.shape)  # (T, SAMPLE_SIZE)

# 2. 切成 70 / 15 / 15（train / val / test）
train_frac = 0.7
val_frac   = 0.15   # test 也是 0.15

cut_train = int(T_total * train_frac)
cut_val   = int(T_total * (train_frac + val_frac))

idx_time = ts_df.index
split_1_time = idx_time[cut_train]
split_2_time = idx_time[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)

train_raw, tmp_raw = ts_all_raw.split_before(split_1_time)
val_raw,   test_raw = tmp_raw.split_before(split_2_time)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

# 3. 用 train 統計量標準化
train_df = ts_df.iloc[:cut_train]

mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)  # 避免除以 0

ts_df_scaled = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

train_ts, tmp_ts = ts_all_scaled.split_before(split_1_time)
val_ts,   test_ts = tmp_ts.split_before(split_2_time)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)
print("\n[Scaled TimeSeries lengths]")
print("T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 真實值（原單位，用來算 MSE/MAE/R²）
vals_all      = ts_df.to_numpy(dtype=np.float32)  # (T_total, SAMPLE_SIZE)
val_true_raw  = vals_all[cut_train:cut_val, :]    # (T_val,   SAMPLE_SIZE)
test_true_raw = vals_all[cut_val:, :]             # (T_test,  SAMPLE_SIZE)

def inverse_scale(x: np.ndarray) -> np.ndarray:
    """
    x: shape (T_horizon, C)
    回傳反標準化後的值，shape 相同
    """
    return x * std_vec.values + mean_vec.values

# ======================================================
# 3.5 自己 loop VAR(p)，建立 AIC / BIC / HQIC / FPE table，並選階
# ======================================================
print("\n[Order selection by manual VAR(p) grid, with AIC/BIC/HQIC/FPE]")

# 月均資料：明確指定時間頻率為每月月初 MS（如果你是月底就改成 freq='M'）
train_scaled_df = ts_df_scaled.iloc[:cut_train].astype("float64").copy()
train_scaled_df.index = pd.date_range(
    start=train_scaled_df.index[0],
    periods=len(train_scaled_df),
    freq="MS",   # 月初頻率；若你的 index 是月底，改成 "M"
)

var_model = VAR(train_scaled_df)

desired_maxlags = 12

# 先找「最大可估的 lag」
max_feasible = None
for p in range(desired_maxlags, 0, -1):
    try:
        _ = var_model.fit(p)
        max_feasible = p
        print(f"最大可估的 lag = {p}")
        break
    except ValueError as e:
        print(f"lag = {p} 估計失敗（{e}），改試小一點的 lag...")

if max_feasible is None:
    raise RuntimeError("在 p=1..12 都無法估計 VAR(p)，可能樣本數太小或資料有問題。")

# 針對 p = 1..max_feasible，一個一個 fit，收集 IC
rows = []
for p in range(1, max_feasible + 1):
    try:
        res_p = var_model.fit(p)
        rows.append({
            "lag":  p,
            "AIC":  float(res_p.aic),
            "BIC":  float(res_p.bic),
            "HQIC": float(res_p.hqic),
            "FPE":  float(res_p.fpe),
        })
        print(f"  p={p} OK: AIC={res_p.aic:.4f}, BIC={res_p.bic:.4f}, "
              f"HQIC={res_p.hqic:.4f}, FPE={res_p.fpe:.4e}")
    except ValueError as e:
        print(f"  p={p} 估計失敗（{e}），停止往更大 p 試。")
        break

ic_table = pd.DataFrame(rows)

# ---- 找出各指標的最佳（最小）值 ----
best_idx_aic  = ic_table["AIC"].idxmin()
best_idx_bic  = ic_table["BIC"].idxmin()
best_idx_hqic = ic_table["HQIC"].idxmin()
best_idx_fpe  = ic_table["FPE"].idxmin()

# 建一個純字串版的 printed_table，避免 dtype 警告
printed_table = pd.DataFrame({
    "lag":  ic_table["lag"].astype(int).astype(str),
    "AIC":  [f"{v:.4f}" for v in ic_table["AIC"]],
    "BIC":  [f"{v:.4f}" for v in ic_table["BIC"]],
    "HQIC": [f"{v:.4f}" for v in ic_table["HQIC"]],
    "FPE":  [f"{v:.4e}" for v in ic_table["FPE"]],
})

# 在最佳位置標星號
printed_table.loc[best_idx_aic,  "AIC"]  += "*"
printed_table.loc[best_idx_bic,  "BIC"]  += "*"
printed_table.loc[best_idx_hqic, "HQIC"] += "*"
printed_table.loc[best_idx_fpe,  "FPE"]  += "*"

print("\nInformation Criteria table (p = 1..{}):".format(int(ic_table['lag'].max())))
print(printed_table.to_string(index=False))

# 兩個不同的階數：BIC 與 AIC
p_bic = int(ic_table.loc[best_idx_bic, "lag"])
p_aic = int(ic_table.loc[best_idx_aic, "lag"])
print(f"\nSelected lag by BIC = {p_bic}")
print(f"Selected lag by AIC = {p_aic}")

# ======================================================
# 4. 用 BIC / AIC 兩種 ORDER 跑 VARIMA，統一流程封裝成 function
# ======================================================
def run_varima_and_metrics(p_order: int,
                           train_ts: TimeSeries,
                           val_ts: TimeSeries,
                           test_ts: TimeSeries,
                           val_true_raw: np.ndarray,
                           test_true_raw: np.ndarray,
                           inverse_scale_fn) -> dict:
    """
    給定 VARIMA 階數 p_order，跑：
      - train -> val 預測
      - train+val -> test 預測
      - 回傳各種指標與預測結果
    """
    ORDER = (p_order, 0, 0)
    print(f"\n[Run VARIMA] ORDER = {ORDER}")

    # ---- train → predict VAL ----
    print("[Step] train → predict VAL")
    model_val = VARIMA(*ORDER)
    model_val.fit(train_ts)

    T_val = len(val_ts)
    pred_val = model_val.predict(n=T_val)
    pred_val_vals = np.asarray(pred_val.all_values(copy=False))
    if pred_val_vals.ndim == 3:
        pred_val_vals = pred_val_vals[..., 0]
    pred_val_raw = inverse_scale_fn(pred_val_vals)

    # ---- train+val → predict TEST ----
    print("[Step] train+val → predict TEST")
    train_val_ts = train_ts.concatenate(val_ts)
    model_test = VARIMA(*ORDER)
    model_test.fit(train_val_ts)

    T_test = len(test_ts)
    pred_test = model_test.predict(n=T_test)
    pred_test_vals = np.asarray(pred_test.all_values(copy=False))
    if pred_test_vals.ndim == 3:
        pred_test_vals = pred_test_vals[..., 0]
    pred_test_raw = inverse_scale_fn(pred_test_vals)

    # ---- 指標 ----
    SAMPLE_SIZE = val_true_raw.shape[1]
    y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
    y_val_pred  = pred_val_raw.reshape(-1, SAMPLE_SIZE)
    y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
    y_test_pred = pred_test_raw.reshape(-1, SAMPLE_SIZE)

    mse_val  = mean_squared_error(y_val_true,  y_val_pred)
    mae_val  = mean_absolute_error(y_val_true, y_val_pred)
    mse_test = mean_squared_error(y_test_true, y_test_pred)
    mae_test = mean_absolute_error(y_test_true, y_test_pred)

    rmse_val  = np.sqrt(mse_val)
    rmse_test = np.sqrt(mse_test)

    r2_val  = r2_score(y_val_true,  y_val_pred)
    r2_test = r2_score(y_test_true, y_test_pred)

    print(f"VAL  RMSE: {rmse_val:.4f}, MSE: {mse_val:.4f}, "
          f"MAE: {mae_val:.4f}, R²: {r2_val:.4f}")
    print(f"TEST RMSE: {rmse_test:.4f}, MSE: {mse_test:.4f}, "
          f"MAE: {mae_test:.4f}, R²: {r2_test:.4f}")

    return {
        "ORDER": ORDER,
        "pred_val_raw":  pred_val_raw,
        "pred_test_raw": pred_test_raw,
        "rmse_val":  rmse_val,
        "mse_val":   mse_val,
        "mae_val":   mae_val,
        "r2_val":    r2_val,
        "rmse_test": rmse_test,
        "mse_test":  mse_test,
        "mae_test":  mae_test,
        "r2_test":   r2_test,
    }

# ======================================================
# 5. 實際跑兩次：BIC 階數 & AIC 階數
# ======================================================
print("\n===== Run VARIMA with BIC-selected order =====")
res_bic = run_varima_and_metrics(
    p_order=p_bic,
    train_ts=train_ts,
    val_ts=val_ts,
    test_ts=test_ts,
    val_true_raw=val_true_raw,
    test_true_raw=test_true_raw,
    inverse_scale_fn=inverse_scale,
)

print("\n===== Run VARIMA with AIC-selected order =====")
res_aic = run_varima_and_metrics(
    p_order=p_aic,
    train_ts=train_ts,
    val_ts=val_ts,
    test_ts=test_ts,
    val_true_raw=val_true_raw,
    test_true_raw=test_true_raw,
    inverse_scale_fn=inverse_scale,
)

# ======================================================
# 6. 彙總比較表
# ======================================================
metrics_compare = pd.DataFrame({
    "Model": ["VARIMA", "VARIMA"],
    "Order": [f"p={p_bic} (BIC)", f"p={p_aic} (AIC)"],
    "Split": ["VAL", "VAL"],
    "RMSE":  [res_bic["rmse_val"], res_aic["rmse_val"]],
    "MSE":   [res_bic["mse_val"],  res_aic["mse_val"]],
    "MAE":   [res_bic["mae_val"],  res_aic["mae_val"]],
    "R2":    [res_bic["r2_val"],   res_aic["r2_val"]],
})

metrics_compare_test = pd.DataFrame({
    "Model": ["VARIMA", "VARIMA"],
    "Order": [f"p={p_bic} (BIC)", f"p={p_aic} (AIC)"],
    "Split": ["TEST", "TEST"],
    "RMSE":  [res_bic["rmse_test"], res_aic["rmse_test"]],
    "MSE":   [res_bic["mse_test"],  res_aic["mse_test"]],
    "MAE":   [res_bic["mae_test"],  res_aic["mae_test"]],
    "R2":    [res_bic["r2_test"],   res_aic["r2_test"]],
})

metrics_compare_all = pd.concat([metrics_compare, metrics_compare_test], ignore_index=True)

print("\n===== VARIMA RESULTS COMPARISON (25 sampled US cells) =====")
print(metrics_compare_all.to_string(index=False))



===== VARIMA baseline on 25 sampled US grid cells =====
Total time steps: 548
ts_df shape: (548, 25)
Train len (raw): 383, Val len: 82, Test len: 83

[Scaled TimeSeries lengths]
T_train = 383 T_val = 82 T_test = 83

[Order selection by manual VAR(p) grid, with AIC/BIC/HQIC/FPE]
最大可估的 lag = 12
  p=1 OK: AIC=-110.8640, BIC=-104.1506, HQIC=-108.2006, FPE=7.1560e-49
  p=2 OK: AIC=-110.6464, BIC=-97.4520, HQIC=-105.4114, FPE=9.2133e-49
  p=3 OK: AIC=-110.0193, BIC=-90.3185, HQIC=-102.2020, FPE=1.8991e-48
  p=4 OK: AIC=-109.3273, BIC=-83.0943, HQIC=-98.9169, FPE=4.6016e-48
  p=5 OK: AIC=-108.4244, BIC=-75.6336, HQIC=-95.4102, FPE=1.5827e-47
  p=6 OK: AIC=-108.3436, BIC=-68.9690, HQIC=-92.7149, FPE=2.9019e-47
  p=7 OK: AIC=-108.0995, BIC=-62.1149, HQIC=-89.8452, FPE=8.1543e-47
  p=8 OK: AIC=-108.3431, BIC=-55.7223, HQIC=-87.4524, FPE=2.0180e-46
  p=9 OK: AIC=-109.7795, BIC=-50.4960, HQIC=-86.2412, FPE=2.5054e-46
  p=10 OK: AIC=-111.6824, BIC=-45.7095, HQIC=-85.4855, FPE=4.0293e-46
  p=11 OK:

/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[Step] train+val → predict TEST


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


VAL  RMSE: 4.7002, MSE: 22.0919, MAE: 3.5191, R²: 0.5981
TEST RMSE: 3.9482, MSE: 15.5882, MAE: 2.8952, R²: 0.6850

===== Run VARIMA with AIC-selected order =====

[Run VARIMA] ORDER = (12, 0, 0)
[Step] train → predict VAL


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/tsa/statespace/varmax.py:373: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[Step] train+val → predict TEST


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/tsa/statespace/varmax.py:373: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


VAL  RMSE: 7.6068, MSE: 57.8629, MAE: 6.0563, R²: -0.0207
TEST RMSE: 7.6600, MSE: 58.6757, MAE: 6.0929, R²: -0.0390

===== VARIMA RESULTS COMPARISON (25 sampled US cells) =====
 Model      Order Split     RMSE       MSE      MAE        R2
VARIMA  p=1 (BIC)   VAL 4.700206 22.091938 3.519129  0.598139
VARIMA p=12 (AIC)   VAL 7.606769 57.862931 6.056346 -0.020728
VARIMA  p=1 (BIC)  TEST 3.948188 15.588186 2.895220  0.684964
VARIMA p=12 (AIC)  TEST 7.660005 58.675671 6.092906 -0.038980


### VARIMA Results Comparison — 25 Sampled US Grid Cells  
(Using All 548 Time Steps)

**Data Summary**  
- Total time steps: **548**  
- Grid cells: **25**  
- Train length: **383**  
- Validation length: **82**  
- Test length: **83**

---

### Information Criteria Summary (p = 1 to 12)

| lag |    AIC    |     BIC     |    HQIC    |      FPE      |
|----:|----------:|------------:|-----------:|--------------:|
| 1   | -110.8640 | **-104.1506** | **-108.2006** | **7.1560e-49** |
| 2   | -110.6464 | -97.4520 | -105.4114 | 9.2133e-49 |
| 3   | -110.0193 | -90.3185 | -102.2020 | 1.8991e-48 |
| 4   | -109.3273 | -83.0943 | -98.9169 | 4.6016e-48 |
| 5   | -108.4244 | -75.6336 | -95.4102 | 1.5827e-47 |
| 6   | -108.3436 | -68.9690 | -92.7149 | 2.9019e-47 |
| 7   | -108.0995 | -62.1149 | -89.8452 | 8.1543e-47 |
| 8   | -108.3431 | -55.7223 | -87.4524 | 2.0180e-46 |
| 9   | -109.7795 | -50.4960 | -86.2412 | 2.5054e-46 |
| 10  | -111.6824 | -45.7095 | -85.4855 | 4.0293e-46 |
| 11  | -115.3493 | -42.6601 | -86.4824 | 3.3583e-46 |
| 12  | **-121.8924** | -42.4600 | -90.3444 | 1.0045e-46 |

- Selected lag by **BIC**: **1**  
- Selected lag by **AIC**: **12**

---

### Performance Comparison

| Model  | Order | Split |   RMSE   |    MSE    |    MAE   |    R²     |
|:------:|:-----:|:-----:|:--------:|:---------:|:--------:|:---------:|
| VARIMA | p=1 (BIC)  | VAL  | 4.700206 | 22.091938 | 3.519129 |  0.598139 |
| VARIMA | p=12 (AIC) | VAL  | 7.606769 | 57.862931 | 6.056346 | -0.020728 |
| VARIMA | p=1 (BIC)  | TEST | 3.948188 | 15.588186 | 2.895220 |  0.684964 |
| VARIMA | p=12 (AIC) | TEST | 7.660005 | 58.675671 | 6.092906 | -0.038980 |


## VARIMA (short time)

In [ ]:
# ===== VARIMA on the 25 sampled US grid cells (with optional shorter time window) =====
import numpy as np
import pandas as pd
from typing import Tuple
from darts import TimeSeries
from darts.models import VARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.api import VAR   # 用來做 VAR(p) 拿 AIC/BIC/HQIC/FPE

print("\n===== VARIMA baseline on 25 sampled US grid cells =====")

# -------------------------------------------------------
# 0. 設定「最多使用的時間點數」(可自行調整 / 或設為 None 用全部)
# -------------------------------------------------------
MAX_T = 200   # 例如只用最後 200 個時間點；若想用全部，改成 None

# 1. 建 target DataFrame：25 個格點 × T 時間點（原始單位）
SAMPLE_SIZE = y_sample.shape[0]   # 預期是 25
T_total_full = y_sample.shape[1]

ts_df_full = pd.DataFrame(
    y_sample.T,  # (T_full, SAMPLE_SIZE)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

# 根據 MAX_T 決定要用多少時間點
if (MAX_T is not None) and (T_total_full > MAX_T):
    # 這裡選擇「保留最後 MAX_T 筆」，你也可以改成 .iloc[:MAX_T]
    ts_df = ts_df_full.iloc[-MAX_T:, :].copy()
    print(f"Use only last {MAX_T} time steps (from {ts_df.index[0]} to {ts_df.index[-1]}).")
else:
    ts_df = ts_df_full.copy()
    print("Use all time steps.")

T_total = len(ts_df)

print("Total time steps used:", T_total)
print("ts_df shape:", ts_df.shape)  # (T_used, SAMPLE_SIZE)

# 2. 切成 70 / 15 / 15（train / val / test）
train_frac = 0.7
val_frac   = 0.15   # test 也是 0.15

cut_train = int(T_total * train_frac)
cut_val   = int(T_total * (train_frac + val_frac))

idx_time = ts_df.index
split_1_time = idx_time[cut_train]
split_2_time = idx_time[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)

train_raw, tmp_raw = ts_all_raw.split_before(split_1_time)
val_raw,   test_raw = tmp_raw.split_before(split_2_time)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

# 3. 用 train 統計量標準化
train_df = ts_df.iloc[:cut_train]

mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)  # 避免除以 0

ts_df_scaled = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

train_ts, tmp_ts = ts_all_scaled.split_before(split_1_time)
val_ts,   test_ts = tmp_ts.split_before(split_2_time)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)
print("\n[Scaled TimeSeries lengths]")
print("T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 真實值（原單位，用來算 MSE/MAE/R²）
vals_all      = ts_df.to_numpy(dtype=np.float32)  # (T_used, SAMPLE_SIZE)
val_true_raw  = vals_all[cut_train:cut_val, :]    # (T_val,   SAMPLE_SIZE)
test_true_raw = vals_all[cut_val:, :]             # (T_test,  SAMPLE_SIZE)

def inverse_scale(x: np.ndarray) -> np.ndarray:
    """
    x: shape (T_horizon, C)
    回傳反標準化後的值，shape 相同
    """
    return x * std_vec.values + mean_vec.values

# ======================================================
# 3.5 自己 loop VAR(p)，建立 AIC / BIC / HQIC / FPE table，並選階
# ======================================================
print("\n[Order selection by manual VAR(p) grid, with AIC/BIC/HQIC/FPE]")

# 月均資料：明確指定時間頻率為每月月初 MS（如果你是月底就改成 freq='M'）
train_scaled_df = ts_df_scaled.iloc[:cut_train].astype("float64").copy()
train_scaled_df.index = pd.date_range(
    start=train_scaled_df.index[0],
    periods=len(train_scaled_df),
    freq="MS",   # 月初頻率；若你的 index 是月底，改成 "M"
)

var_model = VAR(train_scaled_df)

desired_maxlags = 12

# 先找「最大可估的 lag」
max_feasible = None
for p in range(desired_maxlags, 0, -1):
    try:
        _ = var_model.fit(p)
        max_feasible = p
        print(f"最大可估的 lag = {p}")
        break
    except ValueError as e:
        print(f"lag = {p} 估計失敗（{e}），改試小一點的 lag...")

if max_feasible is None:
    raise RuntimeError("在 p=1..12 都無法估計 VAR(p)，可能樣本數太小或資料有問題。")

# 針對 p = 1..max_feasible，一個一個 fit，收集 IC
rows = []
for p in range(1, max_feasible + 1):
    try:
        res_p = var_model.fit(p)
        rows.append({
            "lag":  p,
            "AIC":  float(res_p.aic),
            "BIC":  float(res_p.bic),
            "HQIC": float(res_p.hqic),
            "FPE":  float(res_p.fpe),
        })
        print(f"  p={p} OK: AIC={res_p.aic:.4f}, BIC={res_p.bic:.4f}, "
              f"HQIC={res_p.hqic:.4f}, FPE={res_p.fpe:.4e}")
    except ValueError as e:
        print(f"  p={p} 估計失敗（{e}），停止往更大 p 試。")
        break

ic_table = pd.DataFrame(rows)

# ---- 在 table 中用粗體標出每個指標的最佳（最小）值 ----
best_idx_aic  = ic_table["AIC"].idxmin()
best_idx_bic  = ic_table["BIC"].idxmin()
best_idx_hqic = ic_table["HQIC"].idxmin()
best_idx_fpe  = ic_table["FPE"].idxmin()

printed_table = pd.DataFrame({
    "lag":  ic_table["lag"].astype(int).astype(str),
    "AIC":  [f"{v:.4f}" for v in ic_table["AIC"] ],
    "BIC":  [f"{v:.4f}" for v in ic_table["BIC"] ],
    "HQIC": [f"{v:.4f}" for v in ic_table["HQIC"]],
    "FPE":  [f"{v:.4e}" for v in ic_table["FPE"] ],
})

printed_table.loc[best_idx_aic,  "AIC"]  += "*"
printed_table.loc[best_idx_bic,  "BIC"]  += "*"
printed_table.loc[best_idx_hqic, "HQIC"] += "*"
printed_table.loc[best_idx_fpe,  "FPE"]  += "*"

print("\nInformation Criteria table (p = 1..{}):".format(int(ic_table['lag'].max())))
print(printed_table.to_string(index=False))

# 用 BIC 選 lag（這裡還是用數值 ic_table）
p_selected = int(ic_table.loc[best_idx_bic, "lag"])
print(f"\nSelected lag by BIC = {p_selected}")

ORDER = (p_selected, 0, 0)
print(f"Final VARIMA ORDER = {ORDER}")

# ======================================================
# 4. VARIMA：train → predict val（用選出的 ORDER）
# ======================================================
print(f"\n[Step] VARIMA: train → predict VAL with ORDER={ORDER}")

model_varima_val = VARIMA(*ORDER)
model_varima_val.fit(train_ts)

pred_var_val = model_varima_val.predict(n=T_val)   # 預測 val 長度
pred_var_val_vals = np.asarray(pred_var_val.all_values(copy=False))
if pred_var_val_vals.ndim == 3:
    pred_var_val_vals = pred_var_val_vals[..., 0]

pred_var_val_raw = inverse_scale(pred_var_val_vals)   # (T_val, SAMPLE_SIZE)

# ======================================================
# 5. VARIMA：train+val → predict test（同一個 ORDER）
# ======================================================
print(f"[Step] VARIMA: train+val → predict TEST with ORDER={ORDER}")

train_val_ts = train_ts.concatenate(val_ts)

model_varima_test = VARIMA(*ORDER)
model_varima_test.fit(train_val_ts)

pred_var_test = model_varima_test.predict(n=T_test)
pred_var_test_vals = np.asarray(pred_var_test.all_values(copy=False))
if pred_var_test_vals.ndim == 3:
    pred_var_test_vals = pred_var_test_vals[..., 0]

pred_var_test_raw = inverse_scale(pred_var_test_vals)  # (T_test, SAMPLE_SIZE)

# ======================================================
# 6. 計算 VAL / TEST 的 MSE / MAE / RMSE / R²
# ======================================================
y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
y_val_pred  = pred_var_val_raw.reshape(-1, SAMPLE_SIZE)
y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
y_test_pred = pred_var_test_raw.reshape(-1, SAMPLE_SIZE)

mse_var_val  = mean_squared_error(y_val_true,  y_val_pred)
mae_var_val  = mean_absolute_error(y_val_true, y_val_pred)
mse_var_test = mean_squared_error(y_test_true, y_test_pred)
mae_var_test = mean_absolute_error(y_test_true, y_test_pred)

rmse_var_val  = np.sqrt(mse_var_val)
rmse_var_test = np.sqrt(mse_var_test)

r2_var_val  = r2_score(y_val_true,  y_val_pred)
r2_var_test = r2_score(y_test_true, y_test_pred)

print("\n===== VARIMA RESULTS (25 sampled US cells) =====")
print(f"VAL  RMSE: {rmse_var_val:.4f}, MSE: {mse_var_val:.4f}, "
      f"MAE: {mae_var_val:.4f}, R²: {r2_var_val:.4f}")
print(f"TEST RMSE: {rmse_var_test:.4f}, MSE: {mse_var_test:.4f}, "
      f"MAE: {mae_var_test:.4f}, R²: {r2_var_test:.4f}")

metrics_table_varima = pd.DataFrame({
    "Model": ["VARIMA", "VARIMA"],
    "Split": ["VAL",     "TEST"],
    "RMSE":  [rmse_var_val,  rmse_var_test],
    "MSE":   [mse_var_val,   mse_var_test],
    "MAE":   [mae_var_val,   mae_var_test],
    "R2":    [r2_var_val,    r2_var_test],
})

print("\nMSE / MAE / RMSE / R² table (VARIMA, 25 cells):")
print(metrics_table_varima.to_string(index=False))



===== VARIMA baseline on 25 sampled US grid cells =====
Use only last 200 time steps (from 2009-01-01 00:30:00 to 2025-08-01 00:30:00).
Total time steps used: 200
ts_df shape: (200, 25)
Train len (raw): 140, Val len: 30, Test len: 30

[Scaled TimeSeries lengths]
T_train = 140 T_val = 30 T_test = 30

[Order selection by manual VAR(p) grid, with AIC/BIC/HQIC/FPE]
最大可估的 lag = 12
  p=1 OK: AIC=-112.5680, BIC=-98.8456, HQIC=-106.9916, FPE=1.4479e-49
  p=2 OK: AIC=-112.1599, BIC=-85.1146, HQIC=-101.1694, FPE=4.8767e-49
  p=3 OK: AIC=-115.0458, BIC=-74.5498, HQIC=-98.5892, FPE=3.6842e-49
  p=4 OK: AIC=-126.1172, BIC=-72.0403, HQIC=-104.1417, FPE=7.3896e-51
  p=5 估計失敗（10-th leading minor of the array is not positive definite），停止往更大 p 試。

Information Criteria table (p = 1..4):
lag        AIC       BIC       HQIC         FPE
  1  -112.5680 -98.8456* -106.9916*  1.4479e-49
  2  -112.1599  -85.1146  -101.1694  4.8767e-49
  3  -115.0458  -74.5498   -98.5892  3.6842e-49
  4 -126.1172*  -72.0403  -1

/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[Step] VARIMA: train+val → predict TEST with ORDER=(1, 0, 0)


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



===== VARIMA RESULTS (25 sampled US cells) =====
VAL  RMSE: 2.7036, MSE: 7.3094, MAE: 1.9902, R²: 0.8768
TEST RMSE: 2.4030, MSE: 5.7745, MAE: 1.8265, R²: 0.8796

MSE / MAE / RMSE / R² table (VARIMA, 25 cells):
 Model Split     RMSE      MSE      MAE       R2
VARIMA   VAL 2.703583 7.309362 1.990167 0.876788
VARIMA  TEST 2.403019 5.774500 1.826461 0.879585


### VARIMA (short time) +AIC

In [ ]:
# ===== VARIMA on the 25 sampled US grid cells (with optional shorter time window) =====
import numpy as np
import pandas as pd
from typing import Tuple
from darts import TimeSeries
from darts.models import VARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.api import VAR   # 用來做 VAR(p) 拿 AIC/BIC/HQIC/FPE

print("\n===== VARIMA baseline on 25 sampled US grid cells =====")

# -------------------------------------------------------
# 0. 設定「最多使用的時間點數」(可自行調整 / 或設為 None 用全部)
# -------------------------------------------------------
MAX_T = 200   # 例如只用最後 200 個時間點；若想用全部，改成 None

# 1. 建 target DataFrame：25 個格點 × T 時間點（原始單位）
SAMPLE_SIZE = y_sample.shape[0]   # 預期是 25
T_total_full = y_sample.shape[1]

ts_df_full = pd.DataFrame(
    y_sample.T,  # (T_full, SAMPLE_SIZE)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

# 根據 MAX_T 決定要用多少時間點
if (MAX_T is not None) and (T_total_full > MAX_T):
    # 這裡選擇「保留最後 MAX_T 筆」，你也可以改成 .iloc[:MAX_T]
    ts_df = ts_df_full.iloc[-MAX_T:, :].copy()
    print(f"Use only last {MAX_T} time steps (from {ts_df.index[0]} to {ts_df.index[-1]}).")
else:
    ts_df = ts_df_full.copy()
    print("Use all time steps.")

T_total = len(ts_df)

print("Total time steps used:", T_total)
print("ts_df shape:", ts_df.shape)  # (T_used, SAMPLE_SIZE)

# 2. 切成 70 / 15 / 15（train / val / test）
train_frac = 0.7
val_frac   = 0.15   # test 也是 0.15

cut_train = int(T_total * train_frac)
cut_val   = int(T_total * (train_frac + val_frac))

idx_time = ts_df.index
split_1_time = idx_time[cut_train]
split_2_time = idx_time[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)

train_raw, tmp_raw = ts_all_raw.split_before(split_1_time)
val_raw,   test_raw = tmp_raw.split_before(split_2_time)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

# 3. 用 train 統計量標準化
train_df = ts_df.iloc[:cut_train]

mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)  # 避免除以 0

ts_df_scaled = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

train_ts, tmp_ts = ts_all_scaled.split_before(split_1_time)
val_ts,   test_ts = tmp_ts.split_before(split_2_time)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)
print("\n[Scaled TimeSeries lengths]")
print("T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

# 真實值（原單位，用來算 MSE/MAE/R²）
vals_all      = ts_df.to_numpy(dtype=np.float32)  # (T_used, SAMPLE_SIZE)
val_true_raw  = vals_all[cut_train:cut_val, :]    # (T_val,   SAMPLE_SIZE)
test_true_raw = vals_all[cut_val:, :]             # (T_test,  SAMPLE_SIZE)

def inverse_scale(x: np.ndarray) -> np.ndarray:
    """
    x: shape (T_horizon, C)
    回傳反標準化後的值，shape 相同
    """
    return x * std_vec.values + mean_vec.values

# ======================================================
# 3.5 自己 loop VAR(p)，建立 AIC / BIC / HQIC / FPE table，並選階
# ======================================================
print("\n[Order selection by manual VAR(p) grid, with AIC/BIC/HQIC/FPE]")

# 月均資料：明確指定時間頻率為每月月初 MS（如果你是月底就改成 freq='M'）
train_scaled_df = ts_df_scaled.iloc[:cut_train].astype("float64").copy()
train_scaled_df.index = pd.date_range(
    start=train_scaled_df.index[0],
    periods=len(train_scaled_df),
    freq="MS",   # 月初頻率；若你的 index 是月底，改成 "M"
)

var_model = VAR(train_scaled_df)

desired_maxlags = 12

# 先找「最大可估的 lag」
max_feasible = None
for p in range(desired_maxlags, 0, -1):
    try:
        _ = var_model.fit(p)
        max_feasible = p
        print(f"最大可估的 lag = {p}")
        break
    except ValueError as e:
        print(f"lag = {p} 估計失敗（{e}），改試小一點的 lag...")

if max_feasible is None:
    raise RuntimeError("在 p=1..12 都無法估計 VAR(p)，可能樣本數太小或資料有問題。")

# 針對 p = 1..max_feasible，一個一個 fit，收集 IC
rows = []
for p in range(1, max_feasible + 1):
    try:
        res_p = var_model.fit(p)
        rows.append({
            "lag":  p,
            "AIC":  float(res_p.aic),
            "BIC":  float(res_p.bic),
            "HQIC": float(res_p.hqic),
            "FPE":  float(res_p.fpe),
        })
        print(f"  p={p} OK: AIC={res_p.aic:.4f}, BIC={res_p.bic:.4f}, "
              f"HQIC={res_p.hqic:.4f}, FPE={res_p.fpe:.4e}")
    except ValueError as e:
        print(f"  p={p} 估計失敗（{e}），停止往更大 p 試。")
        break

ic_table = pd.DataFrame(rows)

# ---- 在 table 中標出各指標的最佳（最小）值 ----
best_idx_aic  = ic_table["AIC"].idxmin()
best_idx_bic  = ic_table["BIC"].idxmin()
best_idx_hqic = ic_table["HQIC"].idxmin()
best_idx_fpe  = ic_table["FPE"].idxmin()

printed_table = pd.DataFrame({
    "lag":  ic_table["lag"].astype(int).astype(str),
    "AIC":  [f"{v:.4f}" for v in ic_table["AIC"] ],
    "BIC":  [f"{v:.4f}" for v in ic_table["BIC"] ],
    "HQIC": [f"{v:.4f}" for v in ic_table["HQIC"]],
    "FPE":  [f"{v:.4e}" for v in ic_table["FPE"] ],
})

printed_table.loc[best_idx_aic,  "AIC"]  += "*"
printed_table.loc[best_idx_bic,  "BIC"]  += "*"
printed_table.loc[best_idx_hqic, "HQIC"] += "*"
printed_table.loc[best_idx_fpe,  "FPE"]  += "*"

print("\nInformation Criteria table (p = 1..{}):".format(int(ic_table['lag'].max())))
print(printed_table.to_string(index=False))

# 兩種選階：BIC / AIC
p_bic = int(ic_table.loc[best_idx_bic, "lag"])
p_aic = int(ic_table.loc[best_idx_aic, "lag"])
print(f"\nSelected lag by BIC = {p_bic}")
print(f"Selected lag by AIC = {p_aic}")

# ======================================================
# 4. 定義：給定階數 p_order，跑 VARIMA 並回傳指標與預測
# ======================================================
def run_varima_and_metrics(p_order: int,
                           train_ts: TimeSeries,
                           val_ts: TimeSeries,
                           test_ts: TimeSeries,
                           val_true_raw: np.ndarray,
                           test_true_raw: np.ndarray,
                           inverse_scale_fn):
    """
    給定 VARIMA 階數 p_order，跑：
      - train -> val 預測
      - train+val -> test 預測
      - 回傳 VAL / TEST 指標與預測結果
    """
    ORDER = (p_order, 0, 0)
    print(f"\n[Run VARIMA] ORDER = {ORDER}")

    # ---- train → predict VAL ----
    print("[Step] train → predict VAL")
    model_val = VARIMA(*ORDER)
    model_val.fit(train_ts)

    T_val = len(val_ts)
    pred_val = model_val.predict(n=T_val)
    pred_val_vals = np.asarray(pred_val.all_values(copy=False))
    if pred_val_vals.ndim == 3:
        pred_val_vals = pred_val_vals[..., 0]
    pred_val_raw = inverse_scale_fn(pred_val_vals)

    # ---- train+val → predict TEST ----
    print("[Step] train+val → predict TEST")
    train_val_ts = train_ts.concatenate(val_ts)
    model_test = VARIMA(*ORDER)
    model_test.fit(train_val_ts)

    T_test = len(test_ts)
    pred_test = model_test.predict(n=T_test)
    pred_test_vals = np.asarray(pred_test.all_values(copy=False))
    if pred_test_vals.ndim == 3:
        pred_test_vals = pred_test_vals[..., 0]
    pred_test_raw = inverse_scale_fn(pred_test_vals)

    # ---- 指標 ----
    SAMPLE_SIZE = val_true_raw.shape[1]
    y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
    y_val_pred  = pred_val_raw.reshape(-1, SAMPLE_SIZE)
    y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
    y_test_pred = pred_test_raw.reshape(-1, SAMPLE_SIZE)

    mse_val  = mean_squared_error(y_val_true,  y_val_pred)
    mae_val  = mean_absolute_error(y_val_true, y_val_pred)
    mse_test = mean_squared_error(y_test_true, y_test_pred)
    mae_test = mean_absolute_error(y_test_true, y_test_pred)

    rmse_val  = np.sqrt(mse_val)
    rmse_test = np.sqrt(mse_test)

    r2_val  = r2_score(y_val_true,  y_val_pred)
    r2_test = r2_score(y_test_true, y_test_pred)

    print(f"VAL  RMSE: {rmse_val:.4f}, MSE: {mse_val:.4f}, "
          f"MAE: {mae_val:.4f}, R²: {r2_val:.4f}")
    print(f"TEST RMSE: {rmse_test:.4f}, MSE: {mse_test:.4f}, "
          f"MAE: {mae_test:.4f}, R²: {r2_test:.4f}")

    return {
        "ORDER": ORDER,
        "pred_val_raw":  pred_val_raw,
        "pred_test_raw": pred_test_raw,
        "rmse_val":  rmse_val,
        "mse_val":   mse_val,
        "mae_val":   mae_val,
        "r2_val":    r2_val,
        "rmse_test": rmse_test,
        "mse_test":  mse_test,
        "mae_test":  mae_test,
        "r2_test":   r2_test,
    }

# ======================================================
# 5. 實際跑兩次：BIC 階數 & AIC 階數
# ======================================================
print("\n===== Run VARIMA with BIC-selected order =====")
res_bic = run_varima_and_metrics(
    p_order=p_bic,
    train_ts=train_ts,
    val_ts=val_ts,
    test_ts=test_ts,
    val_true_raw=val_true_raw,
    test_true_raw=test_true_raw,
    inverse_scale_fn=inverse_scale,
)

print("\n===== Run VARIMA with AIC-selected order =====")
res_aic = run_varima_and_metrics(
    p_order=p_aic,
    train_ts=train_ts,
    val_ts=val_ts,
    test_ts=test_ts,
    val_true_raw=val_true_raw,
    test_true_raw=test_true_raw,
    inverse_scale_fn=inverse_scale,
)

# 方便後續畫 residual / raw-mean 圖：沿用「BIC 階數」那組預測
pred_var_val_raw  = res_bic["pred_val_raw"]
pred_var_test_raw = res_bic["pred_test_raw"]

# ======================================================
# 6. 彙總比較表
# ======================================================
metrics_compare = pd.DataFrame({
    "Model": ["VARIMA", "VARIMA", "VARIMA", "VARIMA"],
    "Order": [f"p={p_bic} (BIC)", f"p={p_bic} (BIC)",
              f"p={p_aic} (AIC)", f"p={p_aic} (AIC)"],
    "Split": ["VAL", "TEST", "VAL", "TEST"],
    "RMSE":  [res_bic["rmse_val"],  res_bic["rmse_test"],
              res_aic["rmse_val"],  res_aic["rmse_test"]],
    "MSE":   [res_bic["mse_val"],   res_bic["mse_test"],
              res_aic["mse_val"],   res_aic["mse_test"]],
    "MAE":   [res_bic["mae_val"],   res_bic["mae_test"],
              res_aic["mae_val"],   res_aic["mae_test"]],
    "R2":    [res_bic["r2_val"],    res_bic["r2_test"],
              res_aic["r2_val"],    res_aic["r2_test"]],
})

print("\n===== VARIMA RESULTS COMPARISON (25 sampled US cells, possibly truncated to MAX_T) =====")
print(metrics_compare.to_string(index=False))



===== VARIMA baseline on 25 sampled US grid cells =====
Use only last 200 time steps (from 2009-01-01 00:30:00 to 2025-08-01 00:30:00).
Total time steps used: 200
ts_df shape: (200, 25)
Train len (raw): 140, Val len: 30, Test len: 30

[Scaled TimeSeries lengths]
T_train = 140 T_val = 30 T_test = 30

[Order selection by manual VAR(p) grid, with AIC/BIC/HQIC/FPE]
最大可估的 lag = 12
  p=1 OK: AIC=-112.5680, BIC=-98.8456, HQIC=-106.9916, FPE=1.4479e-49
  p=2 OK: AIC=-112.1599, BIC=-85.1146, HQIC=-101.1694, FPE=4.8767e-49
  p=3 OK: AIC=-115.0458, BIC=-74.5498, HQIC=-98.5892, FPE=3.6842e-49
  p=4 OK: AIC=-126.1172, BIC=-72.0403, HQIC=-104.1417, FPE=7.3896e-51
  p=5 估計失敗（10-th leading minor of the array is not positive definite），停止往更大 p 試。

Information Criteria table (p = 1..4):
lag        AIC       BIC       HQIC         FPE
  1  -112.5680 -98.8456* -106.9916*  1.4479e-49
  2  -112.1599  -85.1146  -101.1694  4.8767e-49
  3  -115.0458  -74.5498   -98.5892  3.6842e-49
  4 -126.1172*  -72.0403  -1

/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[Step] train+val → predict TEST


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


VAL  RMSE: 2.7036, MSE: 7.3094, MAE: 1.9902, R²: 0.8768
TEST RMSE: 2.4030, MSE: 5.7745, MAE: 1.8265, R²: 0.8796

===== Run VARIMA with AIC-selected order =====

[Run VARIMA] ORDER = (4, 0, 0)
[Step] train → predict VAL


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[Step] train+val → predict TEST


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


VAL  RMSE: 3.3079, MSE: 10.9419, MAE: 2.4308, R²: 0.7981
TEST RMSE: 2.0991, MSE: 4.4062, MAE: 1.5815, R²: 0.9073

===== VARIMA RESULTS COMPARISON (25 sampled US cells, possibly truncated to MAX_T) =====
 Model     Order Split     RMSE       MSE      MAE       R2
VARIMA p=1 (BIC)   VAL 2.703583  7.309362 1.990167 0.876788
VARIMA p=1 (BIC)  TEST 2.403019  5.774500 1.826461 0.879585
VARIMA p=4 (AIC)   VAL 3.307856 10.941909 2.430836 0.798077
VARIMA p=4 (AIC)  TEST 2.099094  4.406198 1.581459 0.907295


### VARIMA Results Comparison — 25 Sampled US Grid Cells  
(Using Only Last 200 Time Steps)

**Data Summary**  
- Time window used: **last 200 time steps**  
  - From **2009-01-01 00:30:00**  
  - To **2025-08-01 00:30:00**  
- Total time steps used: **200**  
- Train length: **140**  
- Validation length: **30**  
- Test length: **30**  
- Grid cells: **25**

---

### Information Criteria Summary (p = 1 to 4)

| lag |    AIC    |     BIC     |    HQIC    |      FPE      |
|----:|----------:|------------:|-----------:|--------------:|
| 1   | -112.5680 | **-98.8456** | **-106.9916** | 1.4479e-49 |
| 2   | -112.1599 | -85.1146 | -101.1694 | 4.8767e-49 |
| 3   | -115.0458 | -74.5498 | -98.5892 | 3.6842e-49 |
| 4   | **-126.1172** | -72.0403 | -104.1417 | **7.3896e-51** |

- Selected lag by **BIC**: **1**  
- Selected lag by **AIC**: **4**

---

### Performance Comparison

| Model  | Order | Split |   RMSE   |    MSE    |    MAE   |    R²     |
|:------:|:-----:|:-----:|:--------:|:---------:|:--------:|:---------:|
| VARIMA | p=1 (BIC)  | VAL  | 2.703583 | 7.309362 | 1.990167 | 0.876788 |
| VARIMA | p=1 (BIC)  | TEST | 2.403019 | 5.774500 | 1.826461 | 0.879585 |
| VARIMA | p=4 (AIC)  | VAL  | 3.307856 | 10.941909 | 2.430836 | 0.798077 |
| VARIMA | p=4 (AIC)  | TEST | 2.099094 | 4.406198 | 1.581459 | 0.907295 |
